In [1]:
import os, concurrent.futures as cf

In [2]:
import pathlib
import pathlib as P

In [3]:
from tqdm import tqdm

In [4]:
import pandas as pd
import numpy as np

In [5]:
data_root = pathlib.Path("/home/dataset-assist-0/datafile/shaojiangyi/")

In [6]:
def count_seqs(path):
    n = 0
    with open(path, 'rb') as f:
        for line in f:
            if line.startswith(b'>'):
                n += 1
    return path, n - 1  # 减去 query

In [7]:
a3m_dir = pathlib.Path("/home/dataset-assist-0/datafile/shaojiangyi/latence-dataset/sprot_2204_anno_MSA/")

In [8]:
paths = [os.path.join(r, f)
         for r, _, fs in os.walk(a3m_dir)
         for f in fs if f.endswith('.a3m')]

In [9]:
print(len(paths))

79141


In [16]:
with cf.ThreadPoolExecutor(max_workers=5) as ex:
    msa_size = [(path, n) for path, n in tqdm(ex.map(count_seqs, paths, chunksize=200), total=len(paths))]
        # print(f"{n}\t{path}")

100%|████████████████████████████████████████████████████████████████████████████████| 79141/79141 [00:28<00:00, 2745.36it/s]


In [17]:
msa_size_df = pd.DataFrame(msa_size)

In [18]:
msa_size_df[0] = msa_size_df.loc[:, 0].apply(lambda x: os.path.basename(x))

In [19]:
msa_size_df[1] = msa_size_df.loc[:, 1].apply(lambda x: int(x) // 2 - 1)

In [20]:
msa_size_df

,0,1
0,C1QR1_MOUSE.a3m,1210
1,5GT1_PERFR.a3m,729
2,AQP10_HUMAN.a3m,554
3,CHT5B_MEDTR.a3m,860
4,ECD_HUMAN.a3m,312
...,...,...
79136,BUR2_YEAST.a3m,637
79137,PKS3_ARATH.a3m,135
79138,PKS4_ALOAR.a3m,763
79139,PKS4_ARATH.a3m,144


In [21]:
msa_size_df[1].mean()

np.float64(606.4054535575744)

In [22]:
np.quantile(msa_size_df[1], [0.25,0.5,0.75])

array([320., 608., 843.])

In [33]:
tsv_path = pathlib.Path("sprot_2204_anno_a3m_counts.tsv").absolute()

In [34]:
tsv_path.exists()

True

In [35]:
msa_countdf = pd.read_csv(tsv_path, sep="\t", names=["name","counts (query)", "counts"])

In [39]:
msa_countdf["name"] = msa_countdf["name"].apply(lambda x: os.path.basename(x))

In [37]:
# xs = list(msa_countdf.loc[:, "name"])

In [38]:
# for i, x in enumerate(xs):
#     try:
#         n = os.path.basename(x)
#     except Exception:
#         print(i, x)

In [40]:
msa_countdf

,name,counts (query),counts
0,C1QR1_MOUSE.a3m,2423,2422
1,5GT1_PERFR.a3m,1461,1460
2,AQP10_HUMAN.a3m,1112,1111
3,CHT5B_MEDTR.a3m,1724,1723
4,ECD_HUMAN.a3m,627,626
...,...,...,...
79136,BUR2_YEAST.a3m,1278,1277
79137,PKS3_ARATH.a3m,273,272
79138,PKS4_ALOAR.a3m,1530,1529
79139,PKS4_ARATH.a3m,292,291


In [41]:
np.quantile(msa_countdf["counts"],[0.25,0.5,0.75])

array([ 643., 1218., 1689.])

In [42]:
tsv_path = pathlib.Path("sprot_2204_a3m_counts.tsv").absolute()

In [47]:
msa_countdf = pd.read_csv(tsv_path, sep="\t")

In [48]:
msa_countdf

,path,counts (queryl),counts
0,/home/dataset-assist-0/datafile/shaojiangyi/la...,109,108
1,/home/dataset-assist-0/datafile/shaojiangyi/la...,1033,1032
2,/home/dataset-assist-0/datafile/shaojiangyi/la...,2012,2011
3,/home/dataset-assist-0/datafile/shaojiangyi/la...,1360,1359
4,/home/dataset-assist-0/datafile/shaojiangyi/la...,3,2
...,...,...,...
489217,/home/dataset-assist-0/datafile/shaojiangyi/la...,2023,2022
489218,/home/dataset-assist-0/datafile/shaojiangyi/la...,165,164
489219,/home/dataset-assist-0/datafile/shaojiangyi/la...,3319,3318
489220,/home/dataset-assist-0/datafile/shaojiangyi/la...,1800,1799


In [49]:
np.quantile(msa_countdf["counts"], [0.25,0.5,0.75])

array([ 806., 1069., 1478.])